In [141]:
import pandas as pd
from ydata_profiling import ProfileReport

In [142]:
df = pd.read_csv("Absenteeism_at_work_v2.csv", delimiter=";")


# Aplicando módulo para idades e valores de Distancia Negativas negativas 

In [143]:
df.loc[:, 'Age'] = df['Age'].abs()
df.loc[:, 'Distance from Residence to Work'] = df['Distance from Residence to Work'].abs()


# Retirando linhas com idades inconsistentes

In [144]:
df = df[df['Age'] < 100]
df = df[df['Age'] >=18]

# Removendo linha cujo mês estão fora do escopo de 1 à 12

In [145]:
df = df[df['Month of absence'].between(1,12)]

# Trantando colunas Vazias, substituindo por suas medianas

In [146]:
df['Distance from Residence to Work'] = df['Distance from Residence to Work'].fillna(df['Distance from Residence to Work'].median())
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Transportation expense'] = df['Transportation expense'].fillna(df['Transportation expense'].median())
df['Service time'] = df['Service time'].fillna(df['Service time'].median())
df['Work load Average/day '] = df['Work load Average/day '].fillna(df['Work load Average/day '].median())
df['Hit target'] = df['Hit target'].fillna(df['Hit target'].median())
df['Son'] = df['Son'].fillna(df['Son'].median())
df['Pet'] = df['Pet'].fillna(df['Pet'].median())
df['Absenteeism time in hours'] = df['Absenteeism time in hours'].fillna(df['Absenteeism time in hours'].median())
df['Body mass index'] = df['Body mass index'].fillna(df['Body mass index'].median())


# Retirando linhas vazias de dados Categoricos

In [147]:
df.dropna(subset=[
    'Reason for absence', 
    'Day of the week', 
    'Seasons', 
    'Disciplinary failure',
    'Social drinker',
    'Social smoker'
], inplace=True)

# Retirando colunas que não são relacionadas ao problema

In [148]:
df = df.drop(columns=['ID','Education','Weight','Height'])

# Criação da Variável Categórica (Risk_Class) e Divisão de Treino/Teste

In [149]:
import numpy as np

In [150]:
cortes = [-np.inf, 4, 8, np.inf]
class_names = ['0', '1', '2']
df['Risk_Class'] = pd.cut(df['Absenteeism time in hours'], bins=cortes, labels=class_names)

In [151]:
print("--- Distribuição das Classes de Risco ---")
print(df['Risk_Class'].value_counts())

--- Distribuição das Classes de Risco ---
Risk_Class
0    8139
1    3335
2    1992
Name: count, dtype: int64


# Salvando dataset limpo

In [152]:
df.to_csv('absenteismo_limpo.csv', index=False, encoding='utf-8')

# Gerando Gŕaficos para Análise Exploratoria

In [153]:
profile = ProfileReport(df)
profile.to_file("absenteismo_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 52.43it/s]


In [154]:
import matplotlib.pyplot as plt
import seaborn as sns

In [155]:
sns.set_theme(style="whitegrid")

In [156]:
plt.figure(figsize=(12, 8))

<Figure size 1200x800 with 0 Axes>

In [157]:
colunas_numericas = df.select_dtypes(include=['int64', 'float64']).columns
matriz_correlacao = df[colunas_numericas].corr()

In [158]:
sns.heatmap(matriz_correlacao, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlação entre Variáveis', fontsize=16)
plt.tight_layout()
plt.savefig('Matriz_de_Correlação.png')

In [159]:
plt.figure(figsize=(10, 6))
# Agrupa os dados pelo dia da semana e calcula a média de horas de falta
sns.barplot(data=df, x='Day of the week', y='Absenteeism time in hours', errorbar=None, palette='viridis')
plt.title('Média de Horas de Ausência por Dia da Semana', fontsize=16)
plt.xlabel('Dia da Semana (2=Segunda ... 6=Sexta)')
plt.ylabel('Média de Horas de Falta')
plt.tight_layout()
plt.savefig('Media_horas_falta_dia.png')

C:\Users\Turma02\AppData\Local\Temp\ipykernel_23204\2843265510.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='Day of the week', y='Absenteeism time in hours', errorbar=None, palette='viridis')


# Média de Faltas por Número de filhos

In [160]:
plt.figure(figsize=(8, 6))
sns.barplot(data=df, x='Son', y='Absenteeism time in hours', errorbar=None, palette='mako')
plt.title('Impacto do Número de Filhos no Absenteísmo', fontsize=14)
plt.xlabel('Quantidade de Filhos')
plt.ylabel('Média de Horas de Ausência')
plt.tight_layout()

C:\Users\Turma02\AppData\Local\Temp\ipykernel_23204\1790763673.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='Son', y='Absenteeism time in hours', errorbar=None, palette='mako')


In [161]:
plt.savefig('Grafico_Filhos.png')

In [167]:
plt.figure(figsize=(8, 6))
sns.barplot(data=df, x='Pet', y='Absenteeism time in hours', errorbar=None, palette='mako')
plt.title('Impacto do Número de Pets no Absenteísmo', fontsize=14)
plt.xlabel('Quantidade de Pets')
plt.ylabel('Média de Horas de Ausência')
plt.tight_layout()
plt.savefig('Grafico_Pets.png')

C:\Users\Turma02\AppData\Local\Temp\ipykernel_23204\2583012888.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='Pet', y='Absenteeism time in hours', errorbar=None, palette='mako')


In [162]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='Month of absence', y='Absenteeism time in hours', errorbar=None, color='crimson', marker='o', linewidth=2)
plt.title('Sazonalidade: Média de Horas de Ausência por Mês', fontsize=14)
plt.xlabel('Mês (1=Jan a 12=Dez)')
plt.ylabel('Média de Horas de Ausência')
plt.xticks(range(1, 13)) # Garante que todos os 12 meses apareçam no eixo X
plt.tight_layout()

In [163]:
plt.savefig('Grafico_Meses.png')

In [164]:
# A grande diferença está aqui: estimator='sum' (o Pandas faz a soma pra você!)
sns.barplot(data=df, x='Son', y='Absenteeism time in hours', estimator='sum', errorbar=None, palette='mako')

plt.title('Impacto Total (Soma) de Horas de Ausência por Número de Filhos', fontsize=14)
plt.xlabel('Quantidade de Filhos')
plt.ylabel('Soma Total de Horas de Ausência')
plt.tight_layout()

# Salva a primeira imagem e exibe
plt.savefig('Grafico_Filhos_Soma.png')

C:\Users\Turma02\AppData\Local\Temp\ipykernel_23204\592587576.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='Son', y='Absenteeism time in hours', estimator='sum', errorbar=None, palette='mako')
